# Exercise 1 — Predict Taxi Fare
City of Chicago Taxi Trips dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

### Load the data

The **City of Chicago Taxi Trips** dataset is available as a public BigQuery dataset
(`bigquery-public-data.chicago_taxi_trips.taxi_trips`) and also as a CSV export from the
[Chicago Data Portal](https://data.cityofchicago.org/Transportation/Taxi-Trips/wrvz-psew).

Use **one** of the two options below.

In [ ]:
# OPTION A: Load from BigQuery (works in Google Colab)
# Uncomment and run this cell if you are in Colab and have a GCP project.

# from google.colab import auth
# auth.authenticate_user()
# from google.cloud import bigquery
#
# project_id = "YOUR_GCP_PROJECT_ID"   # <-- replace with your GCP project id
# client = bigquery.Client(project=project_id)
#
# query = """
#     SELECT
#         trip_seconds,
#         trip_miles,
#         pickup_community_area,
#         dropoff_community_area,
#         fare
#     FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
#     WHERE fare IS NOT NULL
#       AND trip_seconds IS NOT NULL
#       AND trip_miles IS NOT NULL
#       AND trip_miles > 0
#       AND trip_seconds > 0
#       AND fare > 0
#     LIMIT 200000
# """
# df = client.query(query).to_dataframe()
# df.head()

In [ ]:
# OPTION B: Load from a local/downloaded CSV file
# Download the CSV from https://data.cityofchicago.org/Transportation/Taxi-Trips/wrvz-psew
# and place it in the "dataset" folder as "taxi_trips.csv", then run this cell.

csv_path = "dataset/taxi_trips.csv"   # <-- change path if needed

df = pd.read_csv(
    csv_path,
    usecols=[
        "Trip Seconds",
        "Trip Miles",
        "Pickup Community Area",
        "Dropoff Community Area",
        "Fare",
    ],
)

df.columns = [
    "trip_seconds",
    "trip_miles",
    "pickup_community_area",
    "dropoff_community_area",
    "fare",
]

df.head()

In [ ]:
# Clean the data
df = df.dropna(subset=["trip_seconds", "trip_miles", "fare"])
df = df[(df["trip_seconds"] > 0) & (df["trip_miles"] > 0) & (df["fare"] > 0)]

# Fill missing community areas with a placeholder category
df["pickup_community_area"] = df["pickup_community_area"].fillna(-1).astype(int)
df["dropoff_community_area"] = df["dropoff_community_area"].fillna(-1).astype(int)

print(df.shape)
df.describe()

In [ ]:
# Features and target
X = df[["trip_seconds", "trip_miles", "pickup_community_area", "dropoff_community_area"]]
y = df["fare"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Build a pipeline: one-hot encode community areas
categorical_features = ["pickup_community_area", "dropoff_community_area"]
numeric_features = ["trip_seconds", "trip_miles"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",  # keep numeric_features as-is
)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", LinearRegression()),
])

model.fit(X_train, y_train)

In [ ]:
# Evaluate the model
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE:  {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R^2:  {r2:.3f}")

In [ ]:
# Plot predicted vs actual fares
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Actual vs Predicted Taxi Fare")
plt.tight_layout()
plt.show()

In [ ]:
# Predict fare for a new trip example
new_trip = pd.DataFrame({
    "trip_seconds": [900],
    "trip_miles": [3.5],
    "pickup_community_area": [8],
    "dropoff_community_area": [32],
})

predicted_fare = model.predict(new_trip)
print(f"Predicted fare: ${predicted_fare[0]:.2f}")